In [1]:
import itertools
import json
import sys
from pathlib import Path

import numpy as np
import tensorflow.keras.backend as K
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import SGD

ROOT_DIR = Path.cwd().parents[1]
sys.path.append(str(ROOT_DIR / "src" / "data_preprocessing"))
from normalize_fn import load

2026-01-14 02:28:19.131074: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-14 02:28:19.311875: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-14 02:28:19.386107: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-14 02:28:19.408385: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-14 02:28:19.538806: I tensorflow/core/platform/cpu_feature_guar

In [2]:
norm_options = ['norm', 'tanh', 'tanh_norm']
hidden_options = [
    [4096, 2048],
    [2048, 1024],
    [4096, 2048, 1024],
    [2048, 1024, 512],
    [1024, 1024]]
lr_options = [1e-2, 1e-3, 1e-4, 1e-5]
dropout_options = [(0, 0), (0.2, 0.5)]
hyperparameter_grid = list(itertools.product(
    norm_options, hidden_options, lr_options, dropout_options
))
best_val_loss = np.inf
best_params = None
best_epoch = None

In [3]:
def moving_average(x, n):
    return np.convolve(x, np.ones(n) / n, mode='valid')

In [ ]:
checkpoint_file = ROOT_DIR / "hyperparam_checkpoint.json"

# Load checkpoint if exists (resume from previous run)
start_idx = 0
if checkpoint_file.exists():
    with open(checkpoint_file, "r") as f:
        checkpoint = json.load(f)
    best_val_loss = checkpoint.get("best_val_loss", np.inf)
    best_params = checkpoint.get("best_params", None)
    best_epoch = checkpoint.get("best_epoch", None)
    start_idx = checkpoint.get("last_completed_idx", -1) + 1
    print(f"Resuming from index {start_idx}, best_val_loss so far: {best_val_loss}")

for idx, (norm_type, hidden_layers, lr, (input_do, hidden_do)) in enumerate(hyperparameter_grid):
    if idx < start_idx:
        continue
    X_tr_np, X_val_np, _, _, y_tr_np, y_val_np, _, _ = load(norm=norm_type)
    model = Sequential()
    for i, units in enumerate(hidden_layers):
        if i == 0:
            model.add(Dense(
                units,
                input_shape=(X_tr_np.shape[1],),
                activation='relu',
                kernel_initializer='he_normal'
            ))
            if input_do > 0:
                model.add(Dropout(input_do))
        else:
            model.add(Dense(
                units,
                activation='relu',
                kernel_initializer='he_normal'
            ))
            if hidden_do > 0:
                model.add(Dropout(hidden_do))
    model.add(Dense(1, activation='linear', kernel_initializer='he_normal'))
    model.compile(
        loss='mean_squared_error',
        optimizer=SGD(
            learning_rate=lr,
            momentum=0.5,
        ))
    hist = model.fit(
        X_tr_np, y_tr_np,
        validation_data=(X_val_np, y_val_np),
        epochs=50,
        batch_size=64,
        shuffle=True,
        verbose=1
    )
    val_losses = np.array(hist.history['val_loss'])
    if np.isnan(val_losses).any():
        continue
    ma_losses = moving_average(val_losses, n=25)
    best_ma_loss = np.min(ma_losses)
    best_ma_epoch = np.argmin(ma_losses) + 25
    print(val_losses, ma_losses, best_ma_loss, best_ma_epoch)

    if best_ma_loss < best_val_loss:
        best_val_loss = best_ma_loss
        best_epoch = best_ma_epoch
        best_params = {
            "norm": norm_type,
            "hidden_layers": hidden_layers,
            "learning_rate": lr,
            "input_dropout": input_do,
            "hidden_dropout": hidden_do,
            "epochs": best_ma_epoch }
    # Save checkpoint after each iteration
    checkpoint_data = {
        "last_completed_idx": idx,
        "best_val_loss": float(best_val_loss),
        "best_params": best_params,
        "best_epoch": int(best_epoch) if best_epoch is not None else None
    }
    with open(checkpoint_file, "w") as f:
        json.dump(checkpoint_data, f, indent=2)

    # Clear model to free memory
    del model
    K.clear_session()

I0000 00:00:1768357708.171367     183 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1768357708.409984     183 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1768357708.410015     183 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1768357708.412240     183 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1768357708.412259     183 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:0

In [ ]:
out_file = ROOT_DIR / "best_hyperparams.txt"
with open(out_file, "w") as f:
    for k, v in best_params.items():
        f.write(f"{k}: {v}\n")
    f.write(f"best_val_loss: {best_val_loss}\n")